In [ ]:
import numpy as np
import pandas as pd
import os
import kagglehub

In [ ]:
%cd /kaggle/working/ 
!git clone https://github.com/fulx17/gaussian-splatting --recursive

In [ ]:
%cd /kaggle/working/gaussian-splatting
!python scripts/apply_improved_gs_rasterizer_patch.py
!pip install --no-build-isolation --no-cache-dir --force-reinstall submodules/diff-gaussian-rasterization
!pip install submodules/simple-knn
!pip install submodules/fused-ssim
!pip install plyfile 
!pip install pycolmap
!pip install lpips

In [ ]:
import subprocess
import sys
import torch

assert torch.cuda.is_available(), "Kaggle accelerator must be set to GPU"
subprocess.run([sys.executable, "scripts/apply_improved_gs_rasterizer_patch.py", "--check-only"], check=True)
from diff_gaussian_rasterization import GaussianRasterizationSettings
assert "pixel_weights" in GaussianRasterizationSettings._fields
subprocess.run([sys.executable, "scripts/smoke_test_improved_gs_rasterizer.py"], check=True)
print("Improved-GS rasterizer loaded on", torch.cuda.get_device_name(0))

In [ ]:
!apt-get update && apt-get install -y colmap

In [ ]:
!python preprocess.py --input "/kaggle/input/datasets/xuanph/phase1/phase1/private_set1" --output "/kaggle/working/cleaned_inputs" --subset HCM0249

In [ ]:
import subprocess
import sys
import os

env = os.environ.copy()
env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

cmd = [
    sys.executable,
    "train_full.py",
    "--subset", "HCM0249",
    "--density_control", "improvedgs",
    "--gaussian_budget", "1500000",
    "--use_las", "1",
    "--use_rap", "1",
    "--use_gc", "1",
    "--use_absgrad", "1",
    "--use_eas", "1",
    "--use_mu", "1",
    "--seed", "0",
    "--iterations", "30000"
]

subprocess.run(cmd, env=env, check=True)
print("Training completed successfully.")

In [ ]:
%cd /kaggle/working/gaussian-splatting
!python render_full.py --iterations 30000 --subset HCM0249

In [ ]:
from pathlib import Path
import shutil

subset = ["HCM0249"]
root = Path("/kaggle/working")

for scene in subset:
    model_dir = root / "model_outputs" / scene
    if model_dir.exists():
        shutil.make_archive(
            base_name=str(root / f"{scene}_model"),
            format="zip",
            root_dir=model_dir.parent,
            base_dir=model_dir.name,
        )

    image_dir = root / "image_outputs" / scene
    if image_dir.exists():
        shutil.make_archive(
            base_name=str(root / f"{scene}_image"),
            format="zip",
            root_dir=image_dir.parent,
            base_dir=image_dir.name,
        )

print("Done!")